# 04 · Brute-force reference posteriors (CPU session)

Runs the first `n_reference` test stars (configs/test_seeds.yaml) on 4 cores. Split long runs across sessions with `STARS = "0-99"`, then `"100-199"`.

In [ ]:
# --- Setup: get the code (no project logic in notebooks; everything lives in src/ and scripts/) ---
import os
import subprocess
import sys

REPO_URL = "https://github.com/powershell1/Almanac.git"
BRANCH = "master"
SRC = "/tmp/Almanac"
if os.path.exists("/kaggle/input/almanac-src/pyproject.toml"):
    SRC = "/kaggle/input/almanac-src"  # offline option: repo uploaded as a Dataset
elif not os.path.exists(SRC):
    url = REPO_URL
    try:  # private repo: add a Kaggle Secret GITHUB_TOKEN
        from kaggle_secrets import UserSecretsClient

        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{tok}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, SRC], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zuko==1.6.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", SRC], check=True)
os.chdir(SRC)
print(subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
WORK = "/kaggle/working"
CFG = "configs/killtest.yaml"

In [ ]:
TEST = "/kaggle/input/gdr4-03-simulate/shards/test"
PREV = "/kaggle/input/gdr4-04-reference/shards/reference"  # optional resume
if os.path.exists(PREV):
    os.makedirs(f"{WORK}/shards", exist_ok=True)
    subprocess.run(["cp", "-rn", PREV, f"{WORK}/shards/"], check=True)
STARS = "0-199"

In [ ]:
!python scripts/reference_posteriors.py --config {CFG} --root {WORK} --n-jobs 4 --stars {STARS} --set paths.test_dir={TEST}